# Figma Refactor: LangGraph + DeepSeek V4 Flash 工作流

## 从 vscode.lm 到 LangGraph 状态机的完整迁移解析

本 Notebook 逐步骤展示将 Figma 导出的机翻 React 代码转换为可维护多页应用的完整管线。

### 架构概览
```
extension.ts (VS Code UI 层)
  └─ WorkflowEngine (适配器)
       └─ LangGraph StateGraph (状态机)
            ├─ 每个 Stage → Graph Node
            ├─ 条件边 (retry / skip / abort / auto-heal)
            └─ 内置 Checkpointing
       └─ DeepSeekClient (LLM 调用)
            └─ OpenAI SDK → api.deepseek.com
```

### 迁移要点
| 方面 | 旧架构 | 新架构 |
|------|--------|--------|
| LLM 后端 | `vscode.lm.sendChatRequest()` (Copilot/Gemini) | `callDeepSeek()` (DeepSeek V4 Flash via OpenAI SDK) |
| 状态管理 | 手写 `state.json` + `saveState/loadState` | LangGraph `Annotation.Root` + 内置 reducer |
| 工作流引擎 | 顺序 `for` 循环 + 手写重试逻辑 | `StateGraph` 状态机 + 条件边路由 |
| 重试机制 | `executeStage()` 内 `while` 循环 | LangGraph 节点级错误处理 + autoHeal 条件边 |
| API Key | Copilot 登录 | 环境变量 `DEEPSEEK_API_KEY` 或 VS Code 设置 |



## 1. 项目结构一览

先了解项目的文件和目录结构：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && find src -type f | sort


### 关键文件说明

| 文件 | 用途 |
|------|------|
| `src/extension.ts` | VS Code 命令注册入口 |
| `src/workflow-engine.ts` | LangGraph 适配器，保持旧接口 |
| `src/workflow-state.ts` | LangGraph `Annotation.Root` 状态定义 |
| `src/langgraph-workflow.ts` | StateGraph 构建 + 全部节点 + 条件边 |
| `src/llm/deepseek.ts` | DeepSeek V4 Flash 客户端 |
| `src/prompts/` | 9 个 Prompt 构建器 |
| `src/ast-guard.ts` | esbuild/tsc 语法检查 |
| `src/html-baseline.ts` | HTML 黄金基线提取 |
| `src/linkage-verifier.ts` | 跨页联动契约验证 |
| `test-fixtures/` | E2E 测试用例 |


## 2. 依赖安装

新增三个关键依赖：
- `@langchain/langgraph` — 状态机框架
- `@langchain/core` — LangChain 核心类型
- `openai` — OpenAI 兼容 SDK（调用 DeepSeek）


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && npm ls @langchain/langgraph @langchain/core openai 2>/dev/null | head -5


## 3. DeepSeek LLM 客户端

`src/llm/deepseek.ts` 封装 OpenAI SDK，提供统一的 `callDeepSeek()` 接口。

核心设计：
- 支持流式/非流式两种模式
- 指数退避重试（最多 3 次）
- API Key 来源：环境变量 `DEEPSEEK_API_KEY` → VS Code 配置
- 默认模型：`deepseek-v4-flash`


In [ ]:
%%bash
cat /workspace/project/machinecode_toreact/src/llm/deepseek.ts


### 测试 DeepSeek 连接

先用简单的 Prompt 验证 API 连通性：


In [ ]:
%%bash
DEEPSEEK_API_KEY=YOUR_DEEPSEEK_API_KEY node -e "const { callDeepSeek, validateApiKey } = require('./out/llm/deepseek.js');(async () => {  const ok = await validateApiKey();  console.log('validateApiKey:', ok);  const resp = await callDeepSeek('Respond with exactly: HELLO_WORLD. No reasoning, no extra text.', { maxTokens: 100 });  console.log('Response:', JSON.stringify(resp.trim()));})();"

## 4. LangGraph 状态定义

`src/workflow-state.ts` 使用 `Annotation.Root` 定义状态 Schema。

每个字段都带有 **reducer**（归约器），定义状态更新的方式：
- **数组字段**（`completedStages` 等）：唯一追加合并
- **对象字段**（`stageResults`）：浅合并
- **简单字段**（`autoMode` 等）：最新值覆盖


In [ ]:
%%bash
cat /workspace/project/machinecode_toreact/src/workflow-state.ts


### 状态字段说明

| 字段 | 类型 | 用途 |
|------|------|------|
| `selectedKeys` | `string[]` | 用户选择的阶段顺序 |
| `allStages` | `StageDefinition[]` | 全部阶段定义 |
| `autoMode` | `boolean` | 自动模式（跳过人工确认） |
| `currentStageIndex` | `number` | 当前阶段索引 |
| `completedStages` | `number[]` | 已完成阶段 |
| `failedStages` | `number[]` | 失败阶段 |
| `skippedStages` | `number[]` | 跳过阶段 |
| `workspaceRoot` | `string` | 工作区路径 |
| `stageResults` | `Record<string, StageResult>` | 各阶段结果 |
| `error` | `string | null` | 错误信息 |
| `healAttempts` | `number` | auto-heal 计数 |
| `maxHealRetries` | `number` | 最大 heal 重试次数 |


## 5. LangGraph 工作流图

`src/langgraph-workflow.ts` 定义了完整的 StateGraph。

### 图结构
```
START -> [第一个阶段] -> route -> [下一阶段] -> route -> ... -> finish -> END
                          ^ (autoHeal 当 verify-linkage 失败时)
```

### 节点列表
| 节点 | 对应阶段 | 类型 |
|------|----------|------|
| `scaffold` | 0. 项目脚手架 | LLM |
| `baseline` | 0.5 结构基线 | 分析 |
| `audit` | 1. 代码审计 | LLM |
| `decompose` | 2. 结构分解 | LLM |
| `verifyLinkage` | 2.5 联动验证 | 分析 |
| `extract` | 3. 组件提取 | LLM |
| `style` | 4. 样式提取 | LLM |
| `types` | 5. TypeScript 增强 | LLM |
| `a11y` | 6. 可访问性 | LLM |
| `data` | 7. 数据层分离 | LLM |
| `polish` | 8. 最终清理 | LLM |
| `autoHeal` | 自动修复 | 控制 |
| `finish` | 完成 | 控制 |


In [ ]:
%%bash
cat /workspace/project/machinecode_toreact/src/langgraph-workflow.ts


## 6. 路由与条件边

核心路由函数决定了工作流的执行路径：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && grep -A 10 'function routeToNextStage\|function decideAfterVerify\|function getStageKeyByIndex' src/langgraph-workflow.ts


## 7. WorkflowEngine — LangGraph 适配器

`src/workflow-engine.ts` 被重构为 LangGraph 的适配器，保持与 `extension.ts` 相同的接口。

关键变更：
1. 删除 `initModel()` — 不再调用 `vscode.lm.selectChatModels()`
2. 删除 `callLm()` — 不再调用 `vscode.lm.sendChatRequest()`
3. 删除 `executeStage()` — 阶段的顺序执行逻辑
4. 删除 `autoHealLinkage()` — auto-heal 由 LangGraph 条件边处理
5. 新增 `run()` 调用 `runWorkflow()` LangGraph 入口
6. 保留 `confirmChanges()` — Manual 模式下的 UI 确认
7. 新增 `processPendingChanges()` — 处理暂存变更


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && grep -n 'async run\|validateApiKey\|processPendingChanges\|applyChanges\|confirmChanges' src/workflow-engine.ts | head -10


## 8. Prompt 工程

每个阶段有独立的 Prompt 构建器，位于 `src/prompts/`。所有 Prompt 共享 `BASE_SYSTEM_PROMPT`。

### Base Prompt 包含
- **设计令牌收敛规则**: 颜色/间距平滑映射到 Tailwind 标准值
- **HTML 结构保真约束**: DOM 层级、交互元素、文本不得丢失
- **跨页联动规则**: 导航保真、参数保真、回调保真
- **输出格式约束**: 必须为 `{files: [{path, content}], summary}` 格式的 JSON


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && head -50 src/prompts/base-prompt.ts


## 9. 编译验证

确保 TypeScript 编译无错误，且 LangGraph 图可以正确构建：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && npx tsc --noEmit 2>&1 && echo 'PASS: TypeScript compilation clean'


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && npx tsc -p ./ 2>&1 && echo 'PASS: Compiled to JS'


### LangGraph 图编译验证

以下代码创建 vscode mock，然后编译并调用 LangGraph StateGraph：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact
# Setup vscode mock for headless execution
mkdir -p node_modules/vscode
cat > node_modules/vscode/index.js << "EOF"
module.exports = {
  workspace: { workspaceFolders: [{ uri: { fsPath: "/tmp" } }], getConfiguration: () => ({ get: () => "" }) },
  window: { createOutputChannel: () => ({ appendLine: () => {}, show: () => {} }) },
  Uri: { file: (p) => ({ fsPath: p }) },
  ViewColumn: { Beside: 2 }
};
EOF
echo "vscode mock created"

# Write temp test script
cat > /tmp/test_langgraph.mjs << "TEOF"
import { buildGraph } from '/workspace/project/machinecode_toreact/out/langgraph-workflow.js';
const graph = buildGraph();
const app = graph.compile();
console.log("PASS: LangGraph graph compiled");
const r = await app.invoke({
  selectedKeys: [], allStages: [], autoMode: true, currentStageIndex: 0,
  completedStages: [], failedStages: [], skippedStages: [], workspaceRoot: "/tmp",
  stageResults: {}, error: null, healAttempts: 0, maxHealRetries: 3
});
console.log("PASS: Workflow invoked (empty run)");
console.log("State fields:", Object.keys(r).join(", "));
TEOF
node /tmp/test_langgraph.mjs
rm /tmp/test_langgraph.mjs

## 10. 端到端测试

### 准备测试环境
用 `test-fixtures/` 中的文件创建一个独立的测试工作区：


In [ ]:
%%bash
rm -rf /tmp/e2e-test
mkdir -p /tmp/e2e-test/src
cp /workspace/project/machinecode_toreact/test-fixtures/*.html /tmp/e2e-test/
cp /workspace/project/machinecode_toreact/test-fixtures/*.jsx /tmp/e2e-test/src/
echo '{"name":"e2e-test","private":true}' > /tmp/e2e-test/package.json
echo "Test workspace ready:"
find /tmp/e2e-test -type f | sort


### Step 10.1: Baseline - HTML 结构基线 + 联动契约

分析阶段，不调用 LLM。扫描 HTML 的 DOM 结构，提取交互元素，建立联动契约。


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && DEEPSEEK_API_KEY=YOUR_DEEPSEEK_API_KEY node -e "const { buildHtmlBaseline } = require('./out/html-baseline.js');const { buildLinkageContracts } = require('./out/linkage-verifier.js');(async () => {  const baseline = await buildHtmlBaseline('/tmp/e2e-test');  console.log('=== HTML Baseline ===');  console.log('Files:', baseline.files.length);  for (const f of baseline.files) {    console.log('  ', f.htmlFile, '->', f.matchedJsxFile || '(none)', '[', f.interactiveElements.length, 'elements ]');  }  const contracts = await buildLinkageContracts('/tmp/e2e-test');  console.log('\n=== Contracts ===');  console.log('Total:', contracts.length);  for (const c of contracts.slice(0, 4)) {    console.log('  [' + c.pattern + ']', c.sourceComponent, '->', c.targetComponent || '?');  }})();"

### Step 10.2: Audit - 代码审计 (LLM)

将源码发送给 DeepSeek，生成审计报告。


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && DEEPSEEK_API_KEY=YOUR_DEEPSEEK_API_KEY node -e "const { callDeepSeek } = require('./out/llm/deepseek.js');const { buildAuditPrompt } = require('./out/prompts/stage1-audit.js');const fs = require('fs'); const path = require('path');function readSrc(root) {  const files = []; const walk = (d) => {    for (const e of fs.readdirSync(d, {withFileTypes:true})) {      const fp = path.join(d, e.name);      if (e.isDirectory() && !['node_modules','.git','.figma-stage','out'].includes(e.name)) walk(fp);      else if (e.name.match(/\.(jsx|tsx|ts|json)$/)) files.push(path.relative(root, fp));    }  }; walk(root);  return files.map(f => { try { return '// ' + f + '\n' + fs.readFileSync(path.join(root, f), 'utf-8'); } catch { return ''; } }).join('\n\n');}(async () => {  const src = readSrc('/tmp/e2e-test');  const prompt = buildAuditPrompt(src);  console.log('Prompt:', prompt.length, 'chars');  const resp = await callDeepSeek(prompt, { maxTokens: 8192 });  console.log('Response:', resp.length, 'chars\n');  try {    const p = JSON.parse(resp.match(/\{.*\}/s)?.[0] || resp);    console.log('Summary:', (p.summary || '').slice(0, 200));  } catch(e) { console.log('Parse error, raw:', resp.slice(0, 200)); }})();"

### Step 10.3: Decompose - 结构分解 (核心 LLM)

将单页条件渲染拆分为 React Router 多页结构。


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && DEEPSEEK_API_KEY=YOUR_DEEPSEEK_API_KEY node -e "const { callDeepSeek } = require('./out/llm/deepseek.js');const { buildDecomposePrompt } = require('./out/prompts/stage2-decompose.js');const fs = require('fs'); const path = require('path');function readSrc(root) {  const files = []; const walk = (d) => {    for (const e of fs.readdirSync(d, {withFileTypes:true})) {      const fp = path.join(d, e.name);      if (e.isDirectory() && !['node_modules','.git','.figma-stage','out'].includes(e.name)) walk(fp);      else if (e.name.match(/\.(jsx|tsx|ts|json)$/)) files.push(path.relative(root, fp));    }  }; walk(root);  return files.map(f => { try { return '// ' + f + '\n' + fs.readFileSync(path.join(root, f), 'utf-8'); } catch { return ''; } }).join('\n\n');}(async () => {  const src = readSrc('/tmp/e2e-test');  const auditJson = await fs.promises.readFile('/tmp/e2e-test/.figma-stage/01-audit/audit.json', 'utf-8').catch(() => '{}');  const prompt = buildDecomposePrompt(auditJson, src);  console.log('Prompt:', prompt.length, 'chars');  const resp = await callDeepSeek(prompt, { maxTokens: 16384 });  console.log('Response:', resp.length, 'chars\n');  try {    const p = JSON.parse(resp.match(/\{.*\}/s)?.[0] || resp);    const files = p.files || [];    console.log('Generated files:');    for (const f of files) {      const fp = f.path || f.filePath || 'unknown';      const lines = (f.content || '').split('\n').length;      console.log('  ' + fp + ' (' + lines + ' lines)');      console.log('    ' + (f.content || '').slice(0, 100).replace(/\n/g, '\n    '));    }    console.log('\nSummary:', p.summary || '(none)');  } catch(e) { console.log('Parse error:', e.message); }})();"

### Step 10.4: Verify Linkage - 联动验证

重构后验证所有联动契约是否仍被满足：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && node -e "const { verifyLinkageContracts } = require('./out/linkage-verifier.js');const fs = require('fs');(async () => {  const cp = '/tmp/e2e-test/.figma-stage/00-linkage/contracts-before.json';  if (!fs.existsSync(cp)) { console.log('No contracts found, run baseline first'); return; }  const contracts = JSON.parse(fs.readFileSync(cp, 'utf-8'));  const report = await verifyLinkageContracts('/tmp/e2e-test', contracts);  console.log('Verification:', report.summary.verified + '/' + report.summary.total + ' passed');  if (report.brokenContracts.length) {    console.log('Broken:');    for (const c of report.brokenContracts) console.log('  [' + c.pattern + ']', c.sourceComponent, '->', c.parameter, ':', c.verificationDetail);  }})();"

## 11. 完整管线执行

使用 `runWorkflow()` 一次性执行所有阶段：


In [ ]:
%%bash
cd /workspace/project/machinecode_toreact && DEEPSEEK_API_KEY=YOUR_DEEPSEEK_API_KEY node -e "const { buildGraph, runWorkflow } = require('./out/langgraph-workflow.js');(async () => {  const graph = buildGraph();  const app = graph.compile();  console.log('Graph compiled, ready to run full pipeline.');  console.log('Use: runWorkflow(selectedKeys, allStages, autoMode, workspaceRoot)');})();"

## 12. 常见问题

### Q: DeepSeek 返回的 content 为空？
A: 推理模型特性。`reasoning_content` 包含思考过程。确保 `max_tokens` >= 100。

### Q: LLM 返回 path 而不是 filePath？
A: Prompt 模板使用 `path`，代码期望 `filePath`。已自动归一化处理。

### Q: 为什么需要 mock vscode？
A: `ast-guard.ts` 导入了 `vscode` 模块。在无 VS Code 的 Node.js 中运行需要 mock。

### Q: 如何换模型？
A: 设置 `figma-refactor.deepseekModel` 或修改 `src/llm/deepseek.ts` 的 `BASE_URL`。


## 13. 参考资料

- [LangGraph TypeScript](https://langchain-ai.github.io/langgraphjs/)
- [DeepSeek API](https://platform.deepseek.com/api-docs)
- [OpenAI SDK](https://www.npmjs.com/package/openai)
- 源代码: `src/langgraph-workflow.ts`, `src/workflow-state.ts`, `src/llm/deepseek.ts`
